# Hugging Face Transformers Assignments

## 1. Sentiment Analysis

1. Create a new _nlp_transformers_ environment
2. Launch Jupyter Notebook
3. Read in the movie reviews data set including the VADER sentiment scores (_movie_reviews_sentiment.csv_)
4. Apply sentiment analysis to the _movie_info_ column using transformers
5. Compare the transformers sentiment scores with the VADER sentiment scores

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", None)
import numpy as np

import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8")
import seaborn as sns
%config InlineBackend.figure_format ="svg"

In [ ]:
from google.colab import files
movie_reviews_upload = files.upload()

In [ ]:
df = pd.read_csv("movie_reviews_sentiment.csv")

In [ ]:
df.info(memory_usage=True , show_counts=True)

In [ ]:
df.shape

In [ ]:
df.head(5)

In [ ]:
sns.pairplot(df , corner = True , diag_kind="kde");

In [ ]:
# to activate the GPU:
import torch
#torch.cuda.is_available()

In [ ]:
%%time

from transformers import pipeline, logging

logging.set_verbosity_error()


sentiment_analyzer = pipeline("sentiment-analysis",
                              model = "distilbert/distilbert-base-uncased-finetuned-sst-2-english",
                              device = "cuda",
                              truncation = True,
                              use_fast = True)

torch.set_num_threads(1)

with torch.no_grad():
  sentiment_scores = df["movie_info"].apply(sentiment_analyzer)
sentiment_scores

In [ ]:
sentiment_scores[0][0]["label"]

In [ ]:
sentiment_scores.apply(lambda x : x[0]["label"])

In [ ]:
df["HF_Label"] = sentiment_scores.apply(lambda x : x[0]["label"])
df["HF_Score"] = sentiment_scores.apply(lambda x : x[0]["score"])

In [ ]:
#df["HF_Score"] = sentiment_scores.apply(lambda x: x[0]["score"] if x[0]["label"] == "POSITIVE" else -x[0]["score"])

In [ ]:
df["HF_Sentiment"] = df.apply(lambda row: row["HF_Score"] if row["HF_Label"] == "POSITIVE" else -row["HF_Score"] , axis =1)
# we want to apply it on each row, so we call axis = 1

In [ ]:
#df.columns = [col.title() for col in df.columns]

In [ ]:
df.columns

In [ ]:
df[["sentiment_vader","HF_Label","HF_Sentiment"]].plot(kind = "hist" , alpha = 0.3, bins = 10, color = ["red", "turquoise"])

plt.legend(bbox_to_anchor = (1,1))
plt.show()

In [ ]:
import seaborn as sns
sns.pairplot(df ,
             diag_kind = "kde",
             corner = True,
             hue = "HF_Label",
             palette = "plasma")
sns.despine()

## 2. Named Entity Recognition

1. Read in the children's books data set (_childrens_books.csv_)
2. Apply NER to the Description column
3. Create a list of all named entities
4. Only include the people (PER)
5. _Extra credit:_ Exclude the authors as well

In [ ]:
from google.colab import files
children_books_upload = files.upload()

In [ ]:
book_df = pd.read_csv("childrens_books.csv")
book_df

In [ ]:
from transformers import pipeline

ner_analyzer = pipeline("ner",
                        model = "dbmdz/bert-large-cased-finetuned-conll03-english",
                        device= -1 ,
                        aggregation_strategy = 'SIMPLE')

In [ ]:
ner_analyzer(book_df["Description"][0])

In [ ]:
list(set([entity['word'] for entity in ner_analyzer(book_df["Description"][0])]))

In [ ]:
book_df["Description"].apply(ner_analyzer)

In [ ]:
book_df["Named_Entity"] = book_df["Description"].apply(lambda x : [entity['word'] for entity in ner_analyzer(x)])
book_df["Named_Entity"]

In [ ]:
all_named_entities = list(set(list(book_df["Named_Entity"].explode())))
all_named_entities

In [ ]:
book_df["PER_Entity"] = book_df["Description"].apply(lambda x : [entity['word'] for entity in ner_analyzer(x) if entity['entity_group'] == "PER"])
book_df["PER_Entity"]

In [ ]:
PER_named_entities = list(set(list(book_df["PER_Entity"].explode())))

In [ ]:
PER_named_entities

In [ ]:
Authors = list(set(book_df["Author"].tolist()))
Authors

In [ ]:
named_entities_clean = [entity for entity in PER_named_entities if entity not in Authors]
named_entities_clean

## 3. Zero-Shot Classification

1. Apply zero-shot classification to the Description column using these five categories:
* adventure & fantasy
* animals & nature
* mystery
* humor
* non-fiction
2. Find the number of books in each category and check a few to see if the results make sense

In [ ]:
from transformers import pipeline
classifier = pipeline("zero-shot-classification",
                      model = "facebook/bart-large-mnli",
                      device = 'cuda')

In [ ]:
import torch
torch.cuda.is_available()

In [ ]:
candidate_labels = ["adventure & fantasy" , "animals & nature" , "mystery" , "humor" , "non-fiction"]

In [ ]:
pd.DataFrame(classifier(book_df["Description"].iloc[0] ,candidate_labels))

In [ ]:
classifier(book_df["Description"].iloc[0] , candidate_labels)["labels"][0]

In [ ]:
book_df["Category"] = book_df["Description"].apply(lambda x : classifier(x , candidate_labels)['labels'][0])

In [ ]:
book_df[["Description" , "Category"]]

In [ ]:
book_df.Category.value_counts().plot(kind = "bar", label = "Histogram of Category labels")
plt.legend()
plt.show()

## 4. Text Summarization

1. Apply text summarization to the Description column
2. Review the results to see if they make sense

In [ ]:
from transformers import pipeline

summarizer = pipeline("summarization",
                      model = "facebook/bart-large-cnn",
                      device = 'cuda' #or -1
                      )

In [ ]:
summarizer(book_df["Description"].iloc[0], early_stopping = True)

In [ ]:
summarizer(book_df["Description"].iloc[0] , early_stopping = True, min_length = 20 , max_length = 50)

In [ ]:
summarizer(book_df["Description"].iloc[0] ,
           min_length = 20 ,
           max_length = 50,
           length_penalty = 0.8)[0]['summary_text']

In [ ]:
book_df["Summary_of_Description"] = book_df["Description"].apply(lambda x : summarizer(x,
                                                                                       min_length = 10,
                                                                                       max_length = 50,
                                                                                       early_stopping = True,
                                                                                       length_penalty = 0.8)[0]['summary_text'])

In [ ]:
book_df[["Description","Summary_of_Description","Category"]]

## 5. Document Similarity

1. Turn the Description column into embeddings using feature extraction
2. Compare the cosine similarity of Harry Potter and the Sorcerer’s Stone compared to all other books
3. Return the top 5 most similar books

In [ ]:
from transformers import pipeline

feature_extractor = pipeline("feature-extraction",
                             model = "sentence-transformers/all-MiniLM-L6-v2",
                             device = "cuda")


In [ ]:
feature_extractor(book_df["Description"][0])

In [ ]:
feature_extractor(book_df["Description"][0]).size

In [ ]:
feature_extractor(book_df["Description"][0])[0][0]

In [ ]:
embeddings = book_df["Description"].apply(lambda x: feature_extractor(x)[0][0])

In [ ]:
book_embeddings = np.vstack(embeddings)
book_embeddings

In [ ]:
book_embeddings.shape

In [ ]:
target_book = "Harry Potter and the Sorcerer's Stone"
target_index = int(book_df["Title"][book_df["Title"].str.contains(target_book) == True].index[0])
target_index

In [ ]:
book_embeddings[target_index].shape

In [ ]:
embedding_target = book_embeddings[target_index].reshape(1,-1) #HP: Harry Potter
embedding_target.shape

In [ ]:
book_embeddings.shape

In [ ]:
embedding_target.shape

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
cosine_similarity(embedding_target , book_embeddings).shape

In [ ]:
cosine_similarity(embedding_target , book_embeddings)

In [ ]:
similarity_scores_target = pd.Series(cosine_similarity(embedding_target , book_embeddings).flatten() ,
                                     name = "target_similarity").sort_values(ascending = False)
similarity_scores_target

In [ ]:
similarity_scores_target[:10].plot(kind = "bar")
plt.show()

In [ ]:
book_similarity_scores = pd.concat([book_df[["Title","Description"]], similarity_scores_target], axis = 1)
book_similarity_scores.sort_values(["target_similarity"], ascending=False)